# 02 — Normalization & clustering

Take the QC-passed AnnData, normalize, select HVGs, run PCA/UMAP, and cluster. We deliberately stay with the standard scanpy log-norm pipeline because deconvolution downstream (cell2location) prefers raw counts and we don't want to commit to SCTransform-style residuals on the side that gets passed to it.

We also run a spatially aware clustering check — Leiden on a spatial neighbor graph — to confirm that the gene-expression clusters track tissue architecture.

In [ ]:
sample        = "GSE227469_angiosarcoma_01"
data_dir      = "data"
results_dir   = "results"
n_top_genes   = 3000
n_pcs         = 30
leiden_res    = 0.6

In [ ]:
from pathlib import Path
import scanpy as sc, squidpy as sq
import matplotlib.pyplot as plt

out_dir = Path(results_dir) / "visium" / sample
adata = sc.read_h5ad(out_dir / "adata_qc.h5ad")
adata.layers["counts"] = adata.X.copy()
adata

## Normalize, HVG, dim-reduce

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes,
                            flavor="seurat_v3", layer="counts")
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=n_pcs)
sc.pl.pca_variance_ratio(adata, n_pcs=n_pcs, log=True)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=n_pcs)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=leiden_res, key_added="leiden")

In [ ]:
sc.pl.umap(adata, color=["leiden", "total_counts", "pct_counts_mt"], ncols=3)
sc.pl.spatial(adata, color="leiden", size=1.4, title="Leiden clusters (expression graph)")

## Spatially aware clustering check
Build a spatial neighbor graph and cluster on it directly. If the spatial clustering broadly tracks the expression-graph clustering, we know our clusters aren't just batch artifacts.

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=6)
sc.tl.leiden(adata, resolution=leiden_res, key_added="leiden_spatial",
             adjacency=adata.obsp["spatial_connectivities"])
sc.pl.spatial(adata, color=["leiden", "leiden_spatial"], size=1.4, ncols=2,
              title=["Expression-graph clusters", "Spatial-graph clusters"])

In [ ]:
# Marker genes per expression cluster
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=20)

In [ ]:
# Quick sarcoma-relevant marker panel
markers = {
    "endothelial":   ["PECAM1", "VWF", "CD34"],
    "fibroblast":    ["COL1A1", "COL3A1", "FAP"],
    "smooth_muscle": ["ACTA2", "MYH11"],
    "immune":        ["PTPRC", "CD68", "CD3D"],
    "proliferation": ["MKI67", "TOP2A"],
}
flat = [g for v in markers.values() for g in v if g in adata.var_names]
sc.pl.spatial(adata, color=flat, ncols=3, size=1.3, cmap="magma")

In [ ]:
out_path = out_dir / "adata_normalized.h5ad"
adata.write(out_path)
print("Wrote", out_path)

## Takeaways

- Expression-graph Leiden clusters and spatial-graph Leiden clusters mostly agree; the spatial graph smooths fragmented small clusters into contiguous regions.
- The marker panel makes the cluster identities approximately legible (endothelial vs. fibroblast vs. immune), but spot-level mixing is real — that's why Notebook 03 turns to deconvolution.
- We keep raw counts in `adata.layers['counts']` so cell2location can use them without having to re-load.